# pushshiftreader — ChangeMyView Demo

This notebook walks through the full workflow for extracting and analysing r/ChangeMyView data from Pushshift archives:

1. **Extract** — pull CMV submissions and comments out of the `.zst` archive files
2. **Build trees** — reconstruct nested comment threads
3. **Detect signals** — annotate records with boolean flags (delta awards, mod actions, etc.)
4. **Explore** — browse submissions, comments, and threads via the Python API
5. **DataFrame export** — load everything into pandas for analysis
6. **Graph export** — export conversation and author-interaction networks

---

### Prerequisites

```bash
pip install -e ".[all]"   # from the pushshiftreader repo root
```

You need Pushshift archive files on disk. The extractor expects:

```
/path/to/pushshift_dumps/
├── comments/
│   ├── RC_2013-01.zst
│   └── ...
└── submissions/
    ├── RS_2013-01.zst
    └── ...
```

In [ ]:
# Adjust these two paths before running
ARCHIVE_PATH = "/path/to/pushshift_dumps"   # root folder containing comments/ and submissions/
OUTPUT_PATH  = "./extracted"                 # where extracted data will be written

---
## 1. Extract r/ChangeMyView from the Archives

In [ ]:
from pushshiftreader import SubredditExtractor, setup_logging

setup_logging()   # prints progress to stdout

extractor = SubredditExtractor(
    archive_path=ARCHIVE_PATH,
    output_path=OUTPUT_PATH,
    subreddits=["ChangeMyView"],
    output_format="both",   # write both CSV and compressed JSONL
    workers=-1,             # use all CPU cores; set workers=1 for sequential mode
)

result = extractor.run()
print(result)

Extraction is **resumable**: re-running the cell skips months that already have a `metadata.json`. To force a full re-extraction pass `force=True`.

You can also limit to a date range:

In [ ]:
# Extract only 2020–2023 (optional)
result = extractor.run(start_month="2020-01", end_month="2023-12")

---
## 2. Build Comment Trees

`TreeBuilder` reads the flat `comments.jsonl.gz` files and reconstructs the nested reply structure, writing `threads.jsonl.gz` to each month directory.

In [ ]:
from pushshiftreader import TreeBuilder

builder = TreeBuilder(f"{OUTPUT_PATH}/ChangeMyView")
builder.build_all_months()

---
## 3. Signal Detection

### 3a. Using the built-in CMV preset

`get_detectors('cmv')` returns the **general** signals (stickied comments, mod distinguished, content removed, author deleted, top-level comments, OP replies) **plus** a delta-award detector specific to CMV.

In [ ]:
from pushshiftreader import get_detectors, SignalDetector

detectors = get_detectors('cmv')
print("Detectors in preset:")
for d in detectors:
    print(f"  {d}")

In [ ]:
sd = SignalDetector(
    extracted_path=f"{OUTPUT_PATH}/ChangeMyView",
    detectors=detectors,
)

results = sd.run_all_months()
# results is a dict: {"YYYY-MM": <rows_written>, ...}
print(f"\nTotal signal rows written: {sum(results.values()):,}")

### 3b. Adding a custom detector

You can mix preset detectors with your own. The `depth` parameter tells you how deep in the reply tree the comment sits (0 = direct reply to the submission).

In [ ]:
from pushshiftreader import Detector, RegexDetector

class LongArgumentDetector(Detector):
    """Fires when a comment is a substantive top-level argument (>500 chars)."""
    def detect_comment(self, comment, thread, depth=0):
        return depth == 0 and len(comment.body or '') > 500

custom_detectors = get_detectors('cmv') + [
    LongArgumentDetector("long_top_level"),
    RegexDetector("cites_study", r"\b(study|research|paper|journal|doi)\b",
                  record_type="comment"),
]

sd2 = SignalDetector(
    extracted_path=f"{OUTPUT_PATH}/ChangeMyView",
    detectors=custom_detectors,
)
# sd2.run_all_months()   # uncomment to run with custom detectors

---
## 4. Explore the Data

### 4a. Load the subreddit

In [ ]:
from pushshiftreader import load_subreddit

data = load_subreddit(f"{OUTPUT_PATH}/ChangeMyView")

print(f"Subreddit : {data.subreddit}")
print(f"Months    : {data.months[:5]} ... ({len(data.months)} total)")
print(f"Submissions: {data.metadata.total_submissions:,}")
print(f"Comments   : {data.metadata.total_comments:,}")

### 4b. Browse submissions

In [ ]:
# Print the 10 highest-scoring submissions from a single month
month = data.months[-1]   # most recent extracted month

top_subs = sorted(data.submissions(month), key=lambda s: s.score, reverse=True)[:10]

for sub in top_subs:
    status = "[removed]" if sub.is_removed else ("[deleted]" if sub.is_deleted else "")
    print(f"  {sub.score:>6}  {sub.title[:80]}  {status}")

### 4c. Browse comments

In [ ]:
# 5 highest-scoring comments from the same month
top_comments = sorted(data.comments(month), key=lambda c: c.score, reverse=True)[:5]

for c in top_comments:
    preview = (c.body or '')[:120].replace('\n', ' ')
    print(f"  score={c.score:>5}  u/{c.author}")
    print(f"  {preview}")
    print()

### 4d. Walk a thread

In [ ]:
# Pick the highest-scoring thread from the month and print its comment tree
threads = list(data.threads(month))
best_thread = max(threads, key=lambda t: t.submission.score)

print(f"Thread: {best_thread.submission.title}")
print(f"Score : {best_thread.submission.score}  |  Comments: {best_thread.comment_count}")
print()

for comment, depth in best_thread.walk():
    indent  = "    " * depth
    delta   = "Δ " if ('Δ' in (comment.body or '') or '!delta' in (comment.body or '').lower()) else ""
    preview = (comment.body or '').split('\n')[0][:80]
    print(f"{indent}{delta}[{comment.score:+}] u/{comment.author}: {preview}")
    if depth >= 3:          # don't print too deep in the demo
        print(f"{indent}    ...")
        break

---
## 5. pandas DataFrame Export

### 5a. Comments DataFrame

In [ ]:
import pandas as pd

# Load one month — signals.csv is joined automatically when present
df = data.comments_dataframe(month)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Thread-level features are joined onto every comment row:

| column | description |
|---|---|
| `depth` | Reply depth (0 = top-level) |
| `thread_size` | Total comments in that thread |
| `time_since_submission` | Seconds after the post was made |
| `submission_id/title/score/author` | Metadata from the parent post |

In [ ]:
# Distribution of comment depth
df['depth'].value_counts().sort_index().rename('count')

In [ ]:
# Comments that contain a delta award (if signal detection was run)
if 'delta_awarded' in df.columns:
    delta_comments = df[df['delta_awarded'] == True]
    print(f"Delta-award comments: {len(delta_comments):,} of {len(df):,}")
    print(f"Average depth of delta comments: {delta_comments['depth'].mean():.2f}")
    print(f"Average score of delta comments: {delta_comments['score'].mean():.2f}")
else:
    print("Run SignalDetector first to see delta_awarded column.")

In [ ]:
# Response time analysis — how quickly do top comments arrive?
top_level = df[df['depth'] == 0].copy()
top_level['hours_after_post'] = top_level['time_since_submission'] / 3600

top_level['hours_after_post'].describe().round(1)

In [ ]:
# Score vs depth — do deeper comments score lower?
df.groupby('depth')['score'].agg(['mean', 'median', 'count']).round(2)

### 5b. Submissions DataFrame

In [ ]:
subs_df = data.submissions_dataframe(month)

print(f"Shape: {subs_df.shape}")
subs_df[['title', 'author', 'score', 'num_comments', 'created_utc']].head(5)

In [ ]:
# Removed vs deleted vs live submissions
removed = (subs_df['selftext'] == '[removed]').sum()
deleted = (subs_df['selftext'] == '[deleted]').sum()
live    = len(subs_df) - removed - deleted

print(f"Live    : {live:,}")
print(f"Removed : {removed:,}")
print(f"Deleted : {deleted:,}")

### 5c. Convert a single Thread to a DataFrame

In [ ]:
thread_df = best_thread.to_dataframe()
print(f"Thread has {len(thread_df)} comments across {thread_df['depth'].max()+1} depth levels")
thread_df[['author', 'score', 'depth', 'time_since_submission', 'body']].head(8)

---
## 6. Graph Export

Export the data as node/edge CSVs for tools like Gephi, NetworkX, or igraph.

### 6a. Comment / conversation graph

In [ ]:
import os
os.makedirs("./graphs", exist_ok=True)

stats = data.export_comment_graph("./graphs", month=month)
print(f"Nodes: {stats['nodes']:,}  Edges: {stats['edges']:,}")

# Preview
pd.read_csv("./graphs/comment_graph_nodes.csv").head(5)

In [ ]:
pd.read_csv("./graphs/comment_graph_edges.csv").head(5)

### 6b. Author-interaction graph

In [ ]:
stats = data.export_author_graph("./graphs", month=month)
print(f"Authors: {stats['nodes']:,}  Interactions: {stats['edges']:,}")

author_nodes = pd.read_csv("./graphs/author_graph_nodes.csv")
author_nodes.sort_values('comment_count', ascending=False).head(10)

In [ ]:
# Most frequent reply pairs
author_edges = pd.read_csv("./graphs/author_graph_edges.csv")
author_edges.sort_values('weight', ascending=False).head(10)

### 6c. Load into NetworkX (optional)

In [ ]:
try:
    import networkx as nx

    G = nx.DiGraph()

    for _, row in author_nodes.iterrows():
        G.add_node(row['author'],
                   comment_count=row['comment_count'],
                   total_score=row['total_score'])

    for _, row in author_edges.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['weight'])

    print(f"Nodes: {G.number_of_nodes():,}")
    print(f"Edges: {G.number_of_edges():,}")

    # Top authors by in-degree (most replied-to)
    top_by_indegree = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:5]
    print("\nMost replied-to authors:")
    for author, deg in top_by_indegree:
        print(f"  {author}: in-degree {deg}")

except ImportError:
    print("Install networkx to run this cell: pip install networkx")

---
## 7. Multi-Month Analysis

Load all months at once — the same API works across the full dataset.

In [ ]:
# Load all months into one DataFrame (may take a while for large datasets)
all_comments = data.comments_dataframe()   # month=None → all months

print(f"Total rows: {len(all_comments):,}")
print(f"Months   : {all_comments['month'].nunique()}")

In [ ]:
# Monthly comment volume
monthly = all_comments.groupby('month').size().rename('comment_count')
monthly

In [ ]:
# Delta rate over time (requires signal detection to have been run)
if 'delta_awarded' in all_comments.columns:
    delta_rate = (
        all_comments.groupby('month')['delta_awarded']
        .agg(deltas='sum', total='count')
    )
    delta_rate['rate_pct'] = (delta_rate['deltas'] / delta_rate['total'] * 100).round(2)
    delta_rate

In [ ]:
# Most prolific commenters across the full dataset
all_comments.groupby('author')['id'].count().sort_values(ascending=False).head(10)